## Intro to CRT Problem
The Chinese Remainder Theorem states that if you know the remainder of a number $n$ divided by a bunch of coprime numbers, you can "uniquely" determine (congruence modulo the product of your divisors) the number $n$.

Formally put:

Given $k$ equations of $x$ modulo $n$, what is $x$?
$$ x \bmod n_1 \equiv r_1 $$
$$ x \bmod n_2 \equiv r_2 $$
$$ \vdots $$
$$ x \bmod n_{k-2} \equiv r_{k-2} $$
$$ x \bmod n_{k-1} \equiv r_{k-1} $$

For now we will focus on $k = 2$ equations.

We can model these constraints using Z3:

In [ ]:
# run this cell a couple times to see it work for different values
x = Int('x')
s = Solver()

n_1 = randint(1, 50)
n_2 = randint(1, 50)
r_1 = randint(0, n_1 - 1)
r_2 = randint(0, n_2 - 1)

s.add(x % n_1 == r_1)
s.add(x % n_2 == r_2)

# n_1 and n_2 MUST BE COPRIME for this to work
print(s)
print(s.check())
if s.check() == sat:
    print(s.model())
    print(s.model()[x].as_long() % (n_1*n_2))
else:
    print(f"n_1 = {n_1} and n_2 = {n_2} were not coprime, please try to run this cell again")

## CRT Algorithm
The Chinese Remainder Theorem can find the value of $x$ if $n_1$ and $n_2$ are coprime.

The CRT is as follows:
$$ x \bmod n_1 \equiv r_1 $$
$$ x \bmod n_2 \equiv r_2 $$
$$ \gcd(n_1, n_2) \equiv 1 $$
$$ m_1 n_1 + m_2 n_2 \equiv 1 $$
$$ x \equiv r_1 m_2 n_2 + r_2 m_1 n_1 $$

We can model the CRT problem using constraints in Z3.

### Greatest Common Divisor
First we need to make sure that $n_1$ and $n_2$ are coprime. We can check this by proving the greatest common factor, or the greatest common divisor, of $a$ and $b$ equals 1.

First we need to find some positive integer $d$ that divides both $a$ and $b$. Then, we need to show that $d$ *must* be equal to 1.

To show this using Z3, we can add a constraint that every $d$ that divides $n_1$ and $n_2$ must be equal to 1.

In [ ]:
def coprime(n_1, n_2, solver):
    d = Int('d')

    solver.add(n_1 > 0)
    solver.add(n_2 > 0)
    # solver.add(ForAll(d, Implies(And(d > 0, n_1 % d == ??, n_2 % d == ??), d == 1))) # EDIT THIS LINE
    solver.add(ForAll(d, Implies(And(d > 0, n_1 % d == 0, n_2 % d == 0), d == 1)))

We can test the `coprime` function with some known values:

In [ ]:
# test some values of your own if you want
s = Solver()
n_1, n_2 = Ints("n_1 n_2")
s.add(n_1 == 3)
s.add(n_2 == 9)
coprime(n_1, n_2, s)
print(s.check()) # should print unsat

s = Solver()
n_1, n_2 = Ints("n_1 n_2")
s.add(n_1 == 17)
s.add(n_2 == 21)
coprime(n_1, n_2, s)
print(s.check()) # should print sat

### Using Z3 To Model The CRT Algorithm
Now we can use our `coprime` code and Z3 to model the CRT algorithm.

Refer to the CRT algorithm above to complete the code.

In [ ]:
def crt(n_1, r_1, n_2, r_2, solver):
    x = Int('x') # the x we are trying to find
    m_1, m_2 = Ints('m_1 m_2')

    coprime(n_1, n_2, solver) # gcd(n_1, n_2) = 1

    s.add(r_1 == r_1 % n_1) # normalize r_1 and r_2
    s.add(r_2 == r_2 % n_2) # aka make sure 0 <= r_1 < n_1 and 0 <= r_2 < n_2

    #solver.add(m_1 + m_2) # REPLACE THIS LINE
    #solver.add(x == 0) # REPLACE THIS LINE
    solver.add(m_1 * n_1 + m_2 * n_2 == 1)
    solver.add(x == r_1 * m_2 * n_2 + r_2 * m_1 * n_1)

We can verify the `crt` function with some known values:

In [ ]:
s = Solver()

# x % 3 = 2
# x % 5 = 3
crt(3, 2, 5, 3, s)

print(s.check()) # should print "sat"
print(s.model())
print(s.model()[x] == 8) # should be equal to 8
print()

s = Solver()

# x % 3 = 2
# x % 6 = 3
crt(3, 2, 6, 3, s)
print(s.check()) # should print "unsat"
                 # why does this print "unsat"?

## Proving CRT

Consider our original problem:
Given 2 equations of $x$ modulo $n$, what is $x$?
$$ x \bmod n_1 \equiv r_1 $$
$$ x \bmod n_2 \equiv r_2 $$

We are trying to prove that the CRT finds an $x$ such that the 2 equations above are true. We will do so "by contradiction".

We will add the following additional constraint to the solver:
$$ x \bmod n_1 \not\equiv r_1 $$
<center>or</center>
$$ x \bmod n_2 \not\equiv r_2 $$

If the CRT algorithm does in fact find an $x$ such that $ x \bmod n_1 \not\equiv r_1 $ $or$ $ x \bmod n_2 \not\equiv r_2 $, then the theorem is disproven.
If the solver is unsatisfiable, however, then no $x$ could be found disproving the CRT, therefore proving the correctness of the Chinese Remainder Theorem.

Lets try it out.

In [ ]:
s = Solver()

n_1, n_2 = Ints('n_1 n_2')
r_1, r_2 = Ints('r_1 r_2')

# find x using CRT algorithm and add to solver
crt(n_1, r_1, n_2, r_2, s)

# prove CRT by checking for a contradiction
# s.add(Or(n_1, n_2)) # REPLACE THIS LINE
s.add(Or(x % n_1 != r_1, x % n_2 != r_2))

print(s.check()) # should print unsat